# 0. les biblios

In [ ]:
import pandas as pd

import re

import matplotlib.pyplot as plt

import seaborn as sns

import numpy as np

# 1. Charger monitoring_device_operations.csv

In [ ]:
# Charger les données

df = pd.read_csv("monitoring_device_operations.csv")

In [ ]:
# 1. Dimensions et types de colonnes

print("Dimensions :", df.shape)

print("\nTypes de colonnes :")

df.info()

In [ ]:
# 2. Échantillon des données

print("\nPremières lignes :")

display(df.head())

- 4994 lignes et 11 colonnes.

- Une ligne = une opération mensuelle d'un appareil de monitoring sur un site (`site_code` + `device_reference` + `operation_start`).

- Toutes les colonnes sont en `object`, y compris les mesures et les dates : même signal qu'avant, des formats sont mélangés.

# 2. Doublons

### a- Doublons de lignes

In [ ]:
print("Nombre de lignes strictement dupliquées :", df.duplicated().sum())

display(df[df.duplicated(keep=False)].sort_values('device_operation_id'))

### b- Doublons sur l'identifiant métier



`device_operation_id` devrait être un identifiant unique par opération.

In [ ]:
print("device_operation_id dupliqués :", df['device_operation_id'].duplicated().sum())

display(df[df['device_operation_id'].duplicated(keep=False)].sort_values('device_operation_id'))



# Vérification croisée avec la clé (site, appareil, début d'opération)

cle = ['site_code', 'device_reference', 'operation_start']

print("Couples (site, appareil, début) dupliqués :", df.duplicated(subset=cle).sum())

- Solution

In [ ]:
# On supprime les doublons stricts puis les doublons d'identifiant (on garde la 1ère occurrence)

df = df.drop_duplicates(keep='first')

df = df.drop_duplicates(subset='device_operation_id', keep='first')



# Vérification

print("Nombre de lignes après suppression :", df.shape[0])

print("Doublons stricts restants :", df.duplicated().sum())

print("Doublons d'identifiant restants :", df['device_operation_id'].duplicated().sum())

### c- Doublons de colonnes

In [ ]:
noms_doublons = df.columns[df.columns.duplicated()]

print("Noms de colonnes en double :", noms_doublons.tolist())



contenu_doublons = df.columns[df.T.duplicated()]

print("Colonnes avec un contenu dupliqué :", contenu_doublons.tolist())

In [ ]:
# site_code et device_reference se correspondent-ils parfaitement ?

print("Appareils par site :", df.groupby('site_code')['device_reference'].nunique().value_counts())

print("Sites par appareil :", df.groupby('device_reference')['site_code'].nunique().value_counts())

→ Correspondance 1-pour-1 entre `site_code` et `device_reference` : chaque site n'a qu'un seul appareil dans

ce jeu de données. On garde les deux colonnes : l'une identifie le lieu, l'autre le matériel — utile pour la

traçabilité si un appareil est un jour remplacé sur le même site.

# 3. Formats incohérents

### Les types de chaque colonne :

In [ ]:
print(df.dtypes)

### - Vérifier le format de device_operation_id et device_reference

In [ ]:
print("Format device_operation_id :", df['device_operation_id'].str.replace(r'\d', '9', regex=True).unique())

print("Format device_reference :", df['device_reference'].str.replace(r'[A-Z0-9]', 'X', regex=True).unique())

print("Manquants :", df['device_operation_id'].isna().sum(), df['device_reference'].isna().sum())

→ Formats homogènes (`DOP-XXXXXXX` et `DEV-XXXXXX-X`). Rien à corriger.

### - Vérifier le format de operation_start / operation_end (dates) <--->

In [ ]:
start_dt = pd.to_datetime(df['operation_start'], errors='coerce')

end_dt = pd.to_datetime(df['operation_end'], errors='coerce')



print("operation_start non convertibles :", start_dt.isna().sum() - df['operation_start'].isna().sum())

print("operation_end non convertibles :", end_dt.isna().sum() - df['operation_end'].isna().sum())



# Exemples de formats qui posent problème

mask_bad = end_dt.isna() & df['operation_end'].notna()

print("\nExemples de operation_end non convertibles :")

display(df.loc[mask_bad, ['device_operation_id', 'operation_start', 'operation_end']].head(10))

**Constat :** `operation_start` est toujours au format ISO, mais `operation_end` mélange plusieurs formats :

`31-May-2025 23:59:59`, `2022/11/30 23:59:59`, `30/04/2019 23:59:59`. C'est exactement le cas "format

variable" décrit dans le cours.

- Solution

In [ ]:
# operation_start est toujours au format ISO : on le précise pour éviter que le mode "mixed"

# ne le confonde avec un format day-first (piège classique : 2019-02-01 lu comme le 2 janvier)

df['operation_start'] = pd.to_datetime(df['operation_start'], format='ISO8601', errors='coerce')



# operation_end mélange plusieurs formats (dont certains ambigus type JJ/MM/AAAA) : on laisse

# pandas déduire le format ligne par ligne, avec dayfirst pour lever l'ambiguïté JJ/MM vs MM/JJ

df['operation_end'] = pd.to_datetime(df['operation_end'], format='mixed', errors='coerce', dayfirst=True)



# Vérification

print("operation_start non convertibles :", df['operation_start'].isna().sum())

print("operation_end non convertibles :", df['operation_end'].isna().sum())

print("Lignes où end <= start :", (df['operation_end'] <= df['operation_start']).sum())

print("Types :", df['operation_start'].dtype, df['operation_end'].dtype)

### - Vérifier le format de uptime_pct et acoustic_quality (numériques) <--->

In [ ]:
for col in ['uptime_pct', 'acoustic_quality']:

    s = df[col]

    non_num = s[pd.to_numeric(s, errors='coerce').isna() & s.notna()]

    print(col, "| non convertibles :", len(non_num), "| exemples :", non_num.unique()[:8])

**Constat :** `uptime_pct` a le même problème de virgule décimale que dans `environmental_monthly.csv`.

- Solution

In [ ]:
df['uptime_pct'] = df['uptime_pct'].astype(str).str.replace(',', '.', regex=False)

df['uptime_pct'] = pd.to_numeric(df['uptime_pct'], errors='coerce')

df['acoustic_quality'] = pd.to_numeric(df['acoustic_quality'], errors='coerce')



# Vérification

print(df[['uptime_pct', 'acoustic_quality']].dtypes)

print(df[['uptime_pct', 'acoustic_quality']].isna().sum())

### - Vérifier le format de maintenance_type (catégorielle) <--->

In [ ]:
print(df['maintenance_type'].value_counts(dropna=False))

**Constat :** 5 valeurs pour 3 catégories réelles (`routine inspection` / `ROUTINE INSPECTION` /

`Routine Inspection`). Même correction que pour `land_use_category` dans le fichier écologique.

- Solution

In [ ]:
df['maintenance_type'] = df['maintenance_type'].str.strip().str.lower()



# Vérification

print(df['maintenance_type'].value_counts())

### - Vérifier le format de record_status (catégorielle)

In [ ]:
print(df['record_status'].value_counts(dropna=False))

- Solution

In [ ]:
df['record_status'] = df['record_status'].str.strip().str.lower()

print(df['record_status'].value_counts())

→ Une seule vraie modalité (`closed`) : cette colonne n'apporte aucune information (variance nulle),

elle sera écartée des analyses de corrélation/visualisation mais conservée dans le fichier exporté.

### - Vérifier le format de maintenance_provider (catégorielle)

In [ ]:
print(df['maintenance_provider'].value_counts(dropna=False))

print("Manquants :", df['maintenance_provider'].isna().sum())

# 4. Valeurs manquantes

### a. Constat du problème

In [ ]:
print("Valeurs manquantes par colonne :")

print(df.isna().sum())



print("\nPourcentage de manquants par colonne :")

print((df.isna().sum() / len(df) * 100).round(2))

- `fault_code` : ~96% de manquants

- `maintenance_provider` : 30 manquants (0.6%)



Deux logiques différentes, comme pour `environmental_monthly.csv`.

### b. fault_code : l'absence de code EST une information



Hypothèse : un code de panne n'est renseigné que lors d'une intervention corrective, pas lors d'une

inspection de routine.

In [ ]:
print(pd.crosstab(df['maintenance_type'], df['fault_code'], dropna=False))

→ Preuve par les données : `fault_code` est rempli à 100% pour `corrective maintenance` et

`sensor replacement`, et jamais pour `routine inspection`. Le vide correspond donc à « aucune panne » et

non à une perte d'information.

In [ ]:
df['fault_code'] = df['fault_code'].fillna('NO_FAULT')



# Vérification

print(df['fault_code'].value_counts())

print("Manquants restants :", df['fault_code'].isna().sum())

### c. maintenance_provider : inférence par appareil



Hypothèse : chaque site/appareil est toujours suivi par le même prestataire de maintenance.

In [ ]:
trous = df[df['maintenance_provider'].isna()]

print("Lignes concernées :", len(trous), "sur", trous['site_code'].nunique(), "sites différents")



sites_a_verifier = trous['site_code'].unique()

for site in sites_a_verifier[:10]:

    prestataires_connus = df[(df['site_code'] == site) & (df['maintenance_provider'].notna())]['maintenance_provider'].unique()

    print(site, "->", prestataires_connus)

→ Résultat : chaque site n'a qu'un seul prestataire dans tout le reste du dataset. Hypothèse confirmée.

- Solution

In [ ]:
df['maintenance_provider'] = df.groupby('site_code')['maintenance_provider'].transform(

    lambda s: s.fillna(s.mode()[0]) if not s.mode().empty else s

)



# Vérification

print("Valeurs manquantes restantes :")

print(df.isna().sum())

# 5. Gestion des variables catégorielles

### a. Création de variables temporelles

In [ ]:
df['year'] = df['operation_start'].dt.year

df['month'] = df['operation_start'].dt.month

df['duration_days'] = (df['operation_end'] - df['operation_start']).dt.days



display(df.head())

### b. Variables ordonnées

Aucune variable réellement ordinale : `maintenance_type`, `fault_code` et `maintenance_provider` sont des

catégories sans hiérarchie naturelle. `record_status` n'a qu'une seule modalité.

### c. Encodage des variables nominales : reporté après le merge

Comme pour `environmental_monthly.csv`, ce fichier fait partie du même lot de 6 fichiers qui seront

fusionnés sur `site_code` (et la dimension temporelle). Encoder maintenant risquerait de créer des colonnes

dummy incohérentes d'un fichier à l'autre. On garde donc les catégories nettoyées en texte, et le One-Hot

Encoding de `maintenance_type`, `fault_code`, `maintenance_provider` sera fait une seule fois, après le

merge final.

In [ ]:
df_stats = df.copy()



print(df_stats.dtypes)

display(df_stats.head())

# 6. Voir les outliers (colonnes numériques)

In [ ]:
cols_num = ['uptime_pct', 'acoustic_quality', 'duration_days']

display(df_stats[cols_num].describe().T)

In [ ]:
plt.figure(figsize=(14, 5))

for i, col in enumerate(cols_num, 1):

    plt.subplot(1, 3, i)

    sns.boxplot(y=df_stats[col], color="skyblue")

    plt.title(col)

plt.tight_layout()

plt.show()

In [ ]:
for col in cols_num:

    q1, q3 = df_stats[col].quantile([0.25, 0.75])

    iqr = q3 - q1

    bas, haut = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    n = ((df_stats[col] < bas) | (df_stats[col] > haut)).sum()

    print(f"{col:20s} min={df_stats[col].min():7.1f} max={df_stats[col].max():7.1f} -> {n} points hors moustaches")

### - Justification du traitement des outliers

- Aucune valeur impossible : pourcentages entre 0 et 100, pas de négatif.

- Les `uptime_pct`/`acoustic_quality` bas sont directement expliqués par le type de maintenance :

In [ ]:
print(df_stats.groupby('maintenance_type')[['uptime_pct', 'acoustic_quality']].mean().round(1))

→ Les mois de `corrective maintenance` et surtout `sensor replacement` ont un uptime/qualité acoustique

nettement plus faibles que les inspections de routine : ce sont des réalités opérationnelles (l'appareil est

en panne ou hors service pendant la réparation), pas des erreurs de mesure. **On les conserve.**

# 7. Histogrammes et distributions

In [ ]:
plt.figure(figsize=(14, 5))

for i, col in enumerate(cols_num, 1):

    plt.subplot(1, 3, i)

    sns.histplot(df_stats[col], kde=True, bins=30, color='skyblue')

    plt.title(f"Distribution : {col}")

plt.tight_layout()

plt.show()

In [ ]:
print(df_stats[cols_num].skew().round(2))

### - Analyse exploratoire



- `uptime_pct` et `acoustic_quality` sont fortement asymétriques vers la gauche (skew négatif) : la

majorité des opérations sont de bonne qualité, avec une traîne de valeurs basses liée aux interventions

correctives — comportement de type loi de puissance inversée plutôt qu'une loi normale classique.

- `duration_days` est très concentré (presque toutes les opérations durent ~28-31 jours, un cycle mensuel),

donc peu de variance à interpréter ici.

### - Répartition des variables catégorielles

In [ ]:
plt.figure(figsize=(12, 4))



plt.subplot(1, 2, 1)

sns.countplot(y='maintenance_type', data=df_stats,

              order=df_stats['maintenance_type'].value_counts().index, color='skyblue')

plt.title("Répartition des types de maintenance")



plt.subplot(1, 2, 2)

sns.countplot(y='fault_code', data=df_stats,

              order=df_stats['fault_code'].value_counts().index, color='skyblue')

plt.title("Répartition des codes de panne")



plt.tight_layout()

plt.show()

# 8. Corrélations

In [ ]:
matrice_spearman = df_stats[['uptime_pct', 'acoustic_quality', 'duration_days']].corr(method='spearman')

matrice_pearson = df_stats[['uptime_pct', 'acoustic_quality', 'duration_days']].corr(method='pearson')



display(matrice_spearman.round(2))

In [ ]:
plt.figure(figsize=(6, 5))

sns.heatmap(matrice_spearman, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)

plt.title("Matrice de corrélation de Spearman")

plt.tight_layout()

plt.show()

In [ ]:
sns.pairplot(df_stats[['uptime_pct', 'acoustic_quality', 'duration_days']],

             kind='scatter', plot_kws={'alpha': 0.3, 'color': 'teal', 's': 15})

plt.suptitle("Nuages de points croisés", y=1.02)

plt.show()

### - Analyse des corrélations



- **`uptime_pct` ↔ `acoustic_quality`** : corrélation positive attendue — un appareil qui tombe hors service

enregistre aussi une moins bonne qualité acoustique pendant la même période.

- **`duration_days`** ne corrèle avec rien : la durée d'un cycle d'opération est quasi fixe (mensuelle), ce

n'est pas un facteur explicatif ici.

- Comme précédemment, **corrélation n'est pas causalité** : le lien réel passe par une variable cachée commune

(le type de maintenance en cours), pas par un effet direct de l'une sur l'autre.

# 9. Export du jeu de données nettoyé

Une seule version exportée : nettoyée, mais **non encodée**. L'encodage One-Hot sera fait une seule

fois, plus tard, sur le jeu de données fusionné avec les 5 autres fichiers du lot.

In [ ]:
df_stats.to_csv("monitoring_device_operations_clean.csv", index=False)



print("Dimensions finales :", df_stats.shape)

print("Valeurs manquantes restantes :", df_stats.isna().sum().sum())

# 10. Synthèse du nettoyage



| Problème identifié | Colonnes concernées | Traitement appliqué |

|---|---|---|

| Doublons stricts (2) et doublons d'identifiant | toutes | suppression, on garde la 1ère occurrence |

| Dates au format mixte (`31-May-2025`, `2022/11/30`...) | `operation_end` | conversion `format='mixed'` en `datetime` |

| Séparateur décimal virgule | `uptime_pct` | remplacement `,` → `.` |

| Casse incohérente (5 valeurs pour 3 catégories) | `maintenance_type` | normalisation en minuscules |

| Casse incohérente (3 valeurs pour 1 catégorie) | `record_status` | normalisation en minuscules |

| Manquants "porteurs de sens" (96%) | `fault_code` | remplacés par la modalité `NO_FAULT`, confirmé par le croisement avec `maintenance_type` |

| Manquants isolés (30) | `maintenance_provider` | inférence par site (1 seul prestataire par site) |

| Encodage des catégorielles | `maintenance_type`, `fault_code`, `maintenance_provider` | **reporté** : fait une seule fois après le merge des 6 fichiers |

| Valeurs extrêmes (uptime/qualité bas) | numériques | **conservées**, expliquées par le type de maintenance |



**Biais introduits et assumés :**

- l'inférence de `maintenance_provider` par site suppose qu'un site ne change jamais de prestataire sur

toute la période — vérifié sur les données disponibles, mais reste une hypothèse pour les 30 lignes concernées ;

- `record_status` étant constant (`closed`), la colonne est gardée mais n'apportera rien à l'analyse ;

- comme pour `environmental_monthly.csv`, on garde la 1ère occurrence des doublons après avoir vérifié

qu'ils étaient identiques.